# Task 3.2 (b) — Full Repeated Nested Cross-Validation

This notebook runs the **complete rnCV procedure** with hyperparameter tuning enabled. Assignment specifications:

- **R = 10 repetitions** of the full nCV procedure
- **N = 5 outer folds** (generalisation-performance estimation)
- **K = 3 inner folds** (hyperparameter tuning)
- **Optuna TPE sampler with 50 trials per inner study**
- **Inner-loop optimisation metric: MCC** (single number, threshold-aware, robust to class imbalance)
- **Best HPs across the 3 inner folds selected by mean inner-validation MCC**
- **Per-repetition seeding** (`seed = base_seed + r`) for reproducibility

This produces 50 outer-test scores per algorithm per metric (5 outer folds × 10 repetitions).

In [1]:
# ── Setup ───
import sys, os, time, pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import RepeatedNestedCV, get_estimator_configs, METRIC_NAMES

os.makedirs('../results', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Data

In [2]:
df = pd.read_csv('../data/students_dataset.csv')
X  = df.drop('num', axis=1)
y  = df['num'].values

print(f'Dataset: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Class distribution: {pd.Series(y).value_counts().to_dict()}')

Dataset: 242 samples × 13 features
Class distribution: {0: 131, 1: 111}


## 2. Configure the rnCV Pipeline

In [3]:
# ── Build the rnCV instance ────
configs = get_estimator_configs(seed=42)
print('Algorithms under evaluation:')
for c in configs:
    print(f'  • {c.name}')

rncv = RepeatedNestedCV(
    estimator_configs = configs,
    n_outer           = 5,
    n_inner           = 3,
    n_repetitions     = 10,
    n_optuna_trials   = 50,
    base_seed         = 42,
    inner_metric      = 'mcc',
)

n_algos = len(configs)
n_inner_fits = n_algos * 10 * 5 * 50 * 3   # algorithms × reps × outer × trials × inner folds
n_outer_fits = n_algos * 10 * 5
print(f'\nTotal model fits planned:')
print(f'  • Inner loop : {n_inner_fits:,} (= {n_algos}×10×5×50×3)')
print(f'  • Outer refit: {n_outer_fits:,}   (= {n_algos}×10×5)')
print(f'  • Grand total: {n_inner_fits + n_outer_fits:,}')

Algorithms under evaluation:
  • LogisticRegression
  • GaussianNB
  • LDA
  • RandomForest
  • LightGBM
  • XGBoost
  • CatBoost

Total model fits planned:
  • Inner loop : 52,500 (= 7×10×5×50×3)
  • Outer refit: 350   (= 7×10×5)
  • Grand total: 52,850


## 3. Run the Full rnCV

Progress is reported per repetition.

In [4]:
# ── Run ────
t0 = time.time()
rncv_results = rncv.run(X, y)
elapsed = time.time() - t0

print(f'\nCompleted in {elapsed:.1f} s ({elapsed/60:.1f} min)')
print(f'Results shape: {rncv_results.shape}')
rncv_results.head()

  Repetition 1/10 complete.
  Repetition 2/10 complete.
  Repetition 3/10 complete.
  Repetition 4/10 complete.
  Repetition 5/10 complete.
  Repetition 6/10 complete.
  Repetition 7/10 complete.
  Repetition 8/10 complete.
  Repetition 9/10 complete.
  Repetition 10/10 complete.

Completed in 3625.3 s (60.4 min)
Results shape: (350, 11)


,algorithm,repetition,outer_fold,mcc,auc,pr_auc,balanced_acc,f1,recall,specificity,precision
0,LogisticRegression,0,0,0.594325,0.913043,0.919684,0.797659,0.791667,0.826087,0.769231,0.760000
1,GaussianNB,0,0,0.713091,0.909699,0.902606,0.855351,0.844444,0.826087,0.884615,0.863636
2,LDA,0,0,0.632722,0.921405,0.931336,0.816890,0.808511,0.826087,0.807692,0.791667
3,RandomForest,0,0,0.676128,0.936455,0.939601,0.838629,0.833333,0.869565,0.807692,0.800000
4,LightGBM,0,0,0.632722,0.915552,0.920253,0.816890,0.808511,0.826087,0.807692,0.791667


## 4. Quick Summary

In [5]:
# ── Summary table: median + 95% bootstrap CI per algorithm ──────────────────
rncv_summary = rncv.summary(['mcc', 'auc', 'pr_auc', 'balanced_acc', 'f1'])
print('rnCV (tuned HPs) — median [95% CI]:')
display(rncv_summary)

rnCV (tuned HPs) — median [95% CI]:


,mcc,auc,pr_auc,balanced_acc,f1
algorithm,,,,,
LogisticRegression,"0.627 [0.594, 0.667]","0.891 [0.872, 0.907]","0.896 [0.864, 0.912]","0.811 [0.790, 0.827]","0.792 [0.766, 0.810]"
GaussianNB,"0.642 [0.607, 0.687]","0.892 [0.878, 0.907]","0.886 [0.860, 0.901]","0.817 [0.798, 0.842]","0.800 [0.773, 0.828]"
LDA,"0.632 [0.623, 0.669]","0.895 [0.879, 0.911]","0.894 [0.875, 0.916]","0.815 [0.806, 0.825]","0.796 [0.780, 0.809]"
RandomForest,"0.627 [0.585, 0.673]","0.887 [0.878, 0.907]","0.893 [0.873, 0.904]","0.809 [0.793, 0.825]","0.786 [0.768, 0.800]"
LightGBM,"0.582 [0.524, 0.601]","0.868 [0.849, 0.888]","0.869 [0.854, 0.881]","0.783 [0.758, 0.799]","0.759 [0.735, 0.791]"
XGBoost,"0.628 [0.558, 0.664]","0.888 [0.860, 0.907]","0.880 [0.867, 0.903]","0.799 [0.778, 0.829]","0.785 [0.757, 0.810]"
CatBoost,"0.623 [0.558, 0.670]","0.892 [0.868, 0.907]","0.888 [0.871, 0.915]","0.808 [0.778, 0.831]","0.789 [0.762, 0.813]"


In [6]:
# ── Quick ranked view by median MCC ──────────────────────────────────────────
print('Algorithms ranked by median MCC (tuned HPs):')
ranking = (rncv_results.groupby('algorithm')['mcc']
            .median().sort_values(ascending=False))
for i, (algo, mcc) in enumerate(ranking.items(), 1):
    print(f'  {i}. {algo:<22s}  median MCC = {mcc:.3f}')

Algorithms ranked by median MCC (tuned HPs):
  1. GaussianNB              median MCC = 0.642
  2. LDA                     median MCC = 0.632
  3. XGBoost                 median MCC = 0.628
  4. LogisticRegression      median MCC = 0.627
  5. RandomForest            median MCC = 0.627
  6. CatBoost                median MCC = 0.623
  7. LightGBM                median MCC = 0.582


## 5. Inspect the Selected Hyperparameters

Check which hyperparameter regions Optuna converged on across folds and repetitions. This indicates which HPs are stable for each algorithm.

In [7]:
# ── Convert best_hps_ to a DataFrame ────────────────────────────────────
hp_records = []
for entry in rncv.best_hps_:
    if entry['hp']:                 # skip empty (i.e., algorithms without hp_space)
        hp_records.append({
            'algorithm':  entry['algorithm'],
            'repetition': entry['repetition'],
            'outer_fold': entry['outer_fold'],
            **entry['hp'],
        })
hp_df = pd.DataFrame(hp_records)
print(f'Hyperparameter records: {hp_df.shape[0]} rows')
hp_df.head()

Hyperparameter records: 350 rows


,algorithm,repetition,outer_fold,C,l1_ratio,var_smoothing,solver,shrinkage,n_estimators,max_depth,...,num_leaves,min_child_samples,reg_alpha,reg_lambda,min_child_weight,subsample,colsample_bytree,iterations,depth,l2_leaf_reg
0,LogisticRegression,0,0,0.136265,0.601695,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GaussianNB,0,0,NaN,NaN,1.767017e-10,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LDA,0,0,NaN,NaN,NaN,eigen,0.524756,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RandomForest,0,0,NaN,NaN,NaN,NaN,NaN,250.0,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LightGBM,0,0,NaN,NaN,NaN,NaN,NaN,250.0,4.0,...,38.0,25.0,0.001674,0.9134,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ── Per-algorithm HP statistics (median of selected values) ──────────────────
for algo in hp_df['algorithm'].unique():
    sub = hp_df[hp_df['algorithm'] == algo].drop(columns=['algorithm', 'repetition', 'outer_fold'])
    if sub.empty: continue
    print(f'\n{algo}:')
    for col in sub.columns:
        vals = sub[col].dropna()
        if len(vals) == 0: continue
        if vals.dtype.kind in 'biufc':       # numeric
            print(f'  {col:<22s}: median={vals.median():.4g}  (range {vals.min():.4g} – {vals.max():.4g})')
        else:                                # categorical
            print(f'  {col:<22s}: mode={vals.mode().iloc[0]}  ({vals.value_counts().to_dict()})')


LogisticRegression:
  C                     : median=0.3387  (range 0.05163 – 9.897)
  l1_ratio              : median=0.2569  (range 0.003757 – 0.9968)

GaussianNB:
  var_smoothing         : median=5.524e-10  (range 1.273e-12 – 8.592e-07)

LDA:
  solver                : mode=eigen  ({'eigen': 27, 'lsqr': 23})
  shrinkage             : median=0.1346  (range 0.0007912 – 0.9493)

RandomForest:
  n_estimators          : median=275  (range 100 – 500)
  max_depth             : median=10  (range 3 – 20)
  min_samples_split     : median=10  (range 2 – 20)
  min_samples_leaf      : median=8  (range 1 – 10)
  max_features          : mode=sqrt  ({'sqrt': 27, 'log2': 23})

LightGBM:
  n_estimators          : median=350  (range 100 – 500)
  max_depth             : median=8.5  (range 3 – 12)
  learning_rate         : median=0.01517  (range 0.002171 – 0.2748)
  num_leaves            : median=34  (range 8 – 64)
  min_child_samples     : median=25.5  (range 6 – 30)
  reg_alpha             : median=0.0

## 6. Save Results

In [9]:
# ── Save results ──────────────────────────────────────────────────────────
out_path = '../results/rncv_results.pkl'
with open(out_path, 'wb') as f:
    pickle.dump({
        'results':  rncv_results,
        'summary':  rncv_summary,
        'best_hps': rncv.best_hps_,
        'hp_df':    hp_df,
        'config': {
            'n_outer':         5,
            'n_inner':         3,
            'n_repetitions':   10,
            'n_optuna_trials': 50,
            'base_seed':       42,
            'inner_metric':    'mcc',
            'algorithms':      [c.name for c in configs],
        },
        'runtime_seconds': elapsed,
    }, f)
print(f'✓ Saved rnCV results to {out_path}')

✓ Saved rnCV results to ../results/rncv_results.pkl
